# Tugas Akhir Praktikum Pembelajaran Mesin (TA-06)
## Implementasi Algoritma Q-Learning pada Skenario Diskrit: *Smart EV Charging Scheduler* di Microgrid Tenaga Surya (PV-EV)

* **Nama Mahasiswa:** Sigit Miraj Permana
* **NIM:** 301240037
* **Program Studi:** Teknik Informatika — Universitas Bale Bandung (UNIBBA)
* **Dosen Pengampu:** Mohammad Bayu Anggara, S.Kom., M.Kom.

---

### 1. Latar Belakang & Perumusan Elemen Reinforcement Learning ($S, A, R$)
Berbeda dengan paradigma *Supervised Learning* yang bergantung pada dataset berlabel eksternal, **Reinforcement Learning (RL)** melatih sebuah **Agen** melalui interaksi langsung (*trial-and-error*) dengan **Lingkungan (Environment)** untuk memaksimalkan akumulasi imbalan (*cumulative reward*) jangka panjang.

Pada proyek ini, dibangun lingkungan simulasi diskrit tanpa dataset eksternal yang memodelkan manajemen energi pada **Stasiun Pengisian Kendaraan Listrik Berbasis Panel Surya (*Microgrid PV-EV Charging Station*)**:

1. **Agent (Agen):** Sistem Kontroler Cerdas Stasiun Pengisian EV (*Smart Charging Controller*).
2. **Environment (Lingkungan):** Simulasi operasional microgrid selama 1 episode penuh ($24	ext{ jam}$) yang mencakup dinamika radiasi matahari (kurva Gaussian dengan puncak siang hari), beban dasar perkotaan, status baterai penyimpanan (*Battery SOC*), serta tarif listrik dinamis (*Time-of-Use*).
3. **State Space ($S$) — Total $3 	imes 3 	imes 3 = 27	ext{ States}$:**
   * **Kondisi Daya Bersih (*Net Power*):** `0 = Defisit` ($\le 0	ext{ kW}$), `1 = Surplus Tipis` ($0	ext{–}12	ext{ kW}$), `2 = Surplus Melimpah` ($> 12	ext{ kW}$).
   * **Level Baterai Stasiun (*Battery SOC*):** `0 = Rendah` ($< 35\%$), `1 = Sedang` ($35\%	ext{–}70\%$), `2 = Tinggi` ($> 70\%$).
   * **Tarif Listrik Dinamis (*Time-of-Use*):** `0 = Murah (Off-Peak)`, `1 = Normal (Mid-Peak)`, `2 = Mahal (Peak Load)`.
4. **Action Space ($A$) — Total $3	ext{ Actions}$:**
   * `0: Standby / Tunda Charging` — Menahan pengisian EV saat daya defisit atau tarif mahal, sambil mengisi ulang baterai stasiun dari panel surya.
   * `1: Eco Charging (PV + Baterai)` — Pengisian daya EV moderat ($11	ext{ kW}$) dengan memprioritaskan energi surya dan cadangan baterai.
   * `2: Fast / Full Charging (PV + Grid)` — Pengisian daya EV kecepatan penuh ($22	ext{ kW}$) menggunakan kombinasi panel surya dan jaringan listrik utama.
5. **Reward Function ($R$):**
   * **+18:** *Fast Charging* saat daya surya melimpah dan tarif listrik murah (*Peak Efficiency*).
   * **+15 / +14:** *Eco Charging* saat surplus tipis dengan baterai memadai, atau saat PV melimpah di jam tarif normal/mahal (*Green Arbitrage*).
   * **+12:** *Standby* saat daya defisit atau tarif puncak & baterai kritis (*Optimal Protection* mencegah *grid overload*).
   * **-16:** Penalti berat jika memaksakan *Fast Charging* saat daya defisit atau jam tarif puncak (*Grid Overload / Tariff Spike*).
   * **-12:** Penalti jika memaksakan *Eco Charging* saat daya defisit dan baterai sudah kritis (*Battery Depletion*).
   * **-6:** Penalti *Opportunity Loss* jika memilih *Standby* padahal energi surya melimpah, baterai penuh, dan tarif murah.


### 2. Import Library & Setup Lingkungan Simulasi
Pada tahap ini kita mengimpor pustaka standar (`NumPy`, `Pandas`, `Matplotlib`, `Seaborn`) serta kelas lingkungan `SmartEVChargingEnv` dan agen `QLearningAgent` dari folder `src/`.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.environment import SmartEVChargingEnv
from src.q_learning_agent import QLearningAgent

# Inisialisasi Lingkungan Simulasi Diskrit
env = SmartEVChargingEnv(max_steps=24, seed=42)
print(f"Total Ruang Keadaan (State Space |S|) : {env.n_states} states")
print(f"Total Ruang Aksi (Action Space |A|)   : {env.n_actions} actions")
print(f"Dimensi Matriks Q-Table               : ({env.n_states}, {env.n_actions}) = {env.n_states * env.n_actions} sel")
print("\nContoh Pemetaan Indeks State ke Deskripsi:")
for sample_s in [0, 4, 13, 21, 26]:
    print(f"  State S{sample_s:02d} -> {env.get_state_description(sample_s)}")


### 3. Implementasi & Pelatihan Algoritma Q-Learning (Langkah 2)
Algoritma Q-Learning memperbarui estimasi nilai kualitas aksi $Q(s, a)$ pada setiap langkah waktu menggunakan **Persamaan Bellman**:

$$Q_{\text{baru}}(s, a) \leftarrow (1 - \alpha) Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') \right]$$

Keseimbangan antara **Eksplorasi** (mencoba aksi acak untuk menemukan strategi baru) dan **Eksploitasi** (memilih aksi dengan nilai $Q$ tertinggi) dikendalikan menggunakan strategi **$\epsilon$-greedy** dengan peluruhan eksponensial (*exponential decay*) dari $\epsilon = 1.0$ menuju $\epsilon_{\min} = 0.01$.


In [ ]:
# Konfigurasi Hiperparameter Utama
TOTAL_EPISODES = 3000
ALPHA = 0.8        # Learning Rate
GAMMA = 0.9        # Discount Factor
EPSILON_START = 1.0
EPSILON_MIN = 0.01
DECAY_RATE = 0.002

agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    learning_rate=ALPHA,
    discount_factor=GAMMA,
    epsilon=EPSILON_START,
    min_epsilon=EPSILON_MIN,
    decay_rate=DECAY_RATE,
    seed=42
)

# Jalankan Proses Pelatihan
rewards_history, epsilon_history = agent.train(env, total_episodes=TOTAL_EPISODES)

print("Pelatihan Q-Learning Selesai!")
print(f"Rata-rata Reward 100 Episode Awal  : {np.mean(rewards_history[:100]):.2f}")
print(f"Rata-rata Reward 100 Episode Akhir : {np.mean(rewards_history[-100:]):.2f}")
print(f"Nilai Epsilon pada Episode Terakhir: {epsilon_history[-1]:.4f}")


### 4. Analisis Konvergensi & Kurva Pembelajaran (Langkah 3)
Untuk mengamati proses belajar agen secara jernih tanpa tertutup fluktuasi stokastik per episode, digunakan **Moving Average (Rata-Rata Bergerak) dengan window 100 episode** yang disandingkan dengan kurva penurunan nilai $\epsilon$.


In [ ]:
window_size = 100
rewards_ma = pd.Series(rewards_history).rolling(window=window_size, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(rewards_history, color="#93C5FD", alpha=0.35, linewidth=0.8, label="Reward Mentah per Episode")
ax1.plot(rewards_ma, color="#1D4ED8", linewidth=2.4, label=f"Moving Average ({window_size} Episode)")
ax1.set_title("Kurva Pembelajaran Q-Learning & Peluruhan Epsilon", fontsize=13, fontweight="bold")
ax1.set_xlabel("Episode Pelatihan", fontsize=11)
ax1.set_ylabel("Total Akumulasi Reward (24 Jam)", fontsize=11, color="#1D4ED8")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(epsilon_history, color="#DC2626", linestyle="--", linewidth=1.8, label="Epsilon (Eksplorasi)")
ax2.set_ylabel("Probabilitas Eksplorasi (Epsilon)", fontsize=11, color="#DC2626")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
plt.tight_layout()
plt.show()


#### Interpretasi Kurva Pembelajaran:
1. **Fase Awal (Episode 0 – 400 | Eksplorasi Intens):** Nilai $\epsilon$ masih tinggi ($1.0 \rightarrow 0.45$), sehingga agen banyak mengambil keputusan acak—termasuk memaksakan *Fast Charging* saat daya defisit—yang menghasilkan akumulasi reward rendah/negatif dan fluktuatif.
2. **Fase Menengah (Episode 400 – 1.500 | Transisi & Pembelajaran):** Seiring meluruhnya $\epsilon$, agen mulai mengeksploitasi sel Q-Table yang bernilai positif. Garis *Moving Average* menanjak tajam karena agen mulai menghindari penalti *grid overload*.
3. **Fase Akhir (Episode 1.500 – 3.000 | Konvergensi):** Ketika $\epsilon$ mendekati batas minimum ($0.01$), kurva mendatar (*plateau*) pada kisaran reward tinggi dan stabil, menandakan bahwa Q-Table telah konvergen menuju kebijakan optimal.


### 5. Ekstraksi & Visualisasi Kebijakan Optimal / Optimal Policy (Langkah 4)
Setelah Q-Table stabil, kebijakan optimal $\pi^*(s)$ untuk setiap *state* $s$ diekstrak dengan mengambil indeks aksi yang memiliki nilai $Q$ tertinggi:

$$\pi^*(s) = \arg\max_{a} Q(s, a)$$


In [ ]:
# Tampilkan Tabel Kebijakan Optimal Lengkap (27 States)
policy_df = agent.extract_policy(env)
display(policy_df)

# Visualisasi Grid Kebijakan Optimal 9x3
policy_matrix = np.zeros((9, 3), dtype=int)
annot_matrix = np.empty((9, 3), dtype=object)
row_labels = []
short_actions = ["STANDBY (0)", "ECO CHARGE (1)", "FAST CHARGE (2)"]

for p in range(3):
    for s in range(3):
        r_idx = p * 3 + s
        row_labels.append(f"{env.power_labels[p]} | {env.soc_labels[s]}")
        for t in range(3):
            sid = env.encode_state(p, s, t)
            best_a = int(np.argmax(agent.q_table[sid]))
            policy_matrix[r_idx, t] = best_a
            annot_matrix[r_idx, t] = f"S{sid:02d}: {short_actions[best_a]}"

plt.figure(figsize=(10.5, 6.5))
cmap_policy = sns.color_palette(["#F59E0B", "#10B981", "#2563EB"])
sns.heatmap(
    policy_matrix, annot=annot_matrix, fmt="", cmap=cmap_policy, cbar=False,
    linewidths=1.5, linecolor="white",
    xticklabels=["Tarif Murah", "Tarif Normal", "Tarif Mahal"],
    yticklabels=row_labels,
    annot_kws={"size": 9.5, "weight": "bold", "color": "white"}
)
plt.title("Peta Kebijakan Optimal (Optimal Policy) Agen Q-Learning", fontsize=12.5, fontweight="bold")
plt.xlabel("Tarif Listrik Dinamis (Time-of-Use)")
plt.ylabel("Kombinasi Daya Bersih & Level Baterai (SOC)")
plt.tight_layout()
plt.show()


### 6. Eksperimen Hiperparameter: Dampak Discount Factor ($\gamma$) & Learning Rate ($\alpha$) (Langkah 5)
Pada bagian ini kita membandingkan dua skenario ekstrem untuk membuktikan teori Reinforcement Learning secara empiris:
1. **Perbandingan Discount Factor ($\gamma = 0.90$ vs $\gamma = 0.10$):** Menguji perbedaan perilaku agen yang mempertimbangkan dampak cadangan baterai jangka panjang (*visioner*) melawan agen yang hanya mengejar reward sesaat (*miopik*).
2. **Perbandingan Learning Rate ($\alpha = 0.80$ vs $\alpha = 0.05$):** Menguji kecepatan pembaruan bobot Q-Table hingga mencapai konvergensi.


In [ ]:
# Eksperimen 1: Gamma Tinggi (0.90) vs Gamma Rendah (0.10)
env_g_low = SmartEVChargingEnv(max_steps=24, seed=42)
agent_g_low = QLearningAgent(learning_rate=0.8, discount_factor=0.1, decay_rate=0.002, seed=42)
rewards_g_low, _ = agent_g_low.train(env_g_low, total_episodes=TOTAL_EPISODES)

# Eksperimen 2: Alpha Tinggi (0.80) vs Alpha Rendah (0.05)
env_a_low = SmartEVChargingEnv(max_steps=24, seed=42)
agent_a_low = QLearningAgent(learning_rate=0.05, discount_factor=0.9, decay_rate=0.002, seed=42)
rewards_a_low, _ = agent_a_low.train(env_a_low, total_episodes=TOTAL_EPISODES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Perbandingan Gamma
axes[0].plot(pd.Series(rewards_history).rolling(100, min_periods=1).mean(), color="#1D4ED8", lw=2.2, label="Gamma = 0.90 (Visioner)")
axes[0].plot(pd.Series(rewards_g_low).rolling(100, min_periods=1).mean(), color="#EA580C", lw=2.2, label="Gamma = 0.10 (Miopik)")
axes[0].set_title("Dampak Discount Factor (γ)", fontweight="bold")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Moving Average Reward (100 Ep)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# Plot Perbandingan Alpha
axes[1].plot(pd.Series(rewards_history).rolling(100, min_periods=1).mean(), color="#059669", lw=2.2, label="Alpha = 0.80 (Cepat)")
axes[1].plot(pd.Series(rewards_a_low).rolling(100, min_periods=1).mean(), color="#7C3AED", lw=2.2, label="Alpha = 0.05 (Lambat)")
axes[1].set_title("Dampak Learning Rate (α)", fontweight="bold")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Moving Average Reward (100 Ep)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()
